In [ ]:
### CS231n_Lec07_review_Recurrent_Neural_Networks


import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(7)
torch.set_num_threads(1)

SEQ_LEN = 10
HIDDEN_SIZE = 16
EPOCHS = 40
BATCH_SIZE = 64
NAMES = ("A -> B", "B -> A")



## 1. Generate each sequence along with its reverse.
def make_dataset(num_pairs, seed):
    assert SEQ_LEN >= 4
    generator = torch.Generator().manual_seed(seed)
    forward = torch.zeros(num_pairs, SEQ_LEN, 2)

    for i in range(num_pairs):
        # The first and last time points should always be left as empty inputs.
        positions = torch.randperm(
            SEQ_LEN - 2, generator=generator
        )[:2] + 1
        first, second = positions.sort().values.tolist()
        forward[i, first, 0] = 1.0   # A가 먼저
        forward[i, second, 1] = 1.0  # B가 나중

    reverse = forward.flip(dims=[1])
    x = torch.cat([forward, reverse], dim=0)
    y = torch.cat([
        torch.zeros(num_pairs, dtype=torch.long),
        torch.ones(num_pairs, dtype=torch.long),
    ])
    assert torch.equal(x.sum(dim=1), torch.ones(len(x), 2))
    return x, y


train_x, train_y = make_dataset(400, seed=1)
test_x, test_y = make_dataset(200, seed=2)

# The average input for all samples is the same, at [1/T, 1/T].
assert torch.allclose(
    train_x.mean(dim=1),
    torch.full((len(train_x), 2), 1.0 / SEQ_LEN),
)



## 2. Summarize the same input by taking its average,
## or process it using an RNN.
class MeanClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.head = nn.Linear(2, 2)

    def forward(self, x):
        return self.head(x.mean(dim=1))


class RNNClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.rnn = nn.RNN(
            input_size=2,
            hidden_size=HIDDEN_SIZE,
            batch_first=True,
            nonlinearity="tanh",
        )
        self.head = nn.Linear(HIDDEN_SIZE, 2)

    def forward(self, x):
        outputs, h_n = self.rnn(x)
        return self.head(h_n[-1])



## 3. Perform backpropagation over the entire sequence: full BPTT.
def train_model(model):
    loader = DataLoader(
        TensorDataset(train_x, train_y),
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=torch.Generator().manual_seed(30),
    )
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    criterion = nn.CrossEntropyLoss()
    losses = []

    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0.0
        for batch_x, batch_y in loader:
            optimizer.zero_grad()
            loss = criterion(model(batch_x), batch_y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(batch_x)
        losses.append(total_loss / len(train_x))

    model.eval()
    with torch.no_grad():
        scores = model(test_x)
        accuracy = (scores.argmax(1) == test_y).float().mean().item()
    assert scores.shape == (len(test_x), 2)
    assert torch.isfinite(scores).all()
    return model, losses, accuracy


torch.manual_seed(10)
mean_model, mean_losses, mean_acc = train_model(MeanClassifier())
torch.manual_seed(10)
rnn_model, rnn_losses, rnn_acc = train_model(RNNClassifier())

print(f"Mean pooling | test accuracy = {mean_acc:.3f}")
print(f"RNN          | test accuracy = {rnn_acc:.3f}")



## 4. Verify the direct calculation of internal shapes and hidden states.
example = test_x[:8]
with torch.no_grad():
    outputs, h_n = rnn_model.rnn(example)
    h = torch.zeros(8, HIDDEN_SIZE)
    manual_states = []
    rnn = rnn_model.rnn
    for t in range(SEQ_LEN):
        h = torch.tanh(
            example[:, t] @ rnn.weight_ih_l0.T + rnn.bias_ih_l0
            + h @ rnn.weight_hh_l0.T + rnn.bias_hh_l0
        )
        manual_states.append(h)
    manual_outputs = torch.stack(manual_states, dim=1)

assert outputs.shape == (8, SEQ_LEN, HIDDEN_SIZE)
assert h_n.shape == (1, 8, HIDDEN_SIZE)
assert torch.allclose(outputs[:, -1], h_n[-1], atol=1e-6)
assert torch.allclose(outputs, manual_outputs, atol=1e-5)

print("Input:", tuple(example.shape))
print("All hidden states:", tuple(outputs.shape))
print("Final hidden state:", tuple(h_n.shape))



## 5. Check the predictions for reverse-order pairs involving the same event.
pair = test_x[[0, len(test_x) // 2]]
with torch.no_grad():
    mean_scores = mean_model(pair)
    rnn_predictions = rnn_model(pair).argmax(1)
assert torch.allclose(mean_scores[0], mean_scores[1])

for i in range(2):
    tokens = [
        "A" if row[0] == 1 else "B" if row[1] == 1 else "."
        for row in pair[i].tolist()
    ]
    print(" ".join(tokens), "| RNN:", NAMES[rnn_predictions[i].item()])

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(mean_losses, label="Mean pooling")
axes[0].plot(rnn_losses, label="RNN")
axes[0].set(xlabel="Epoch", ylabel="Cross-entropy", title="Training loss")
axes[0].legend()
axes[0].grid(True)
axes[1].bar(["Mean pooling", "RNN"], [mean_acc, rnn_acc])
axes[1].set(ylim=(0, 1.05), ylabel="Accuracy", title="Test accuracy")
plt.tight_layout()
plt.show()